In [26]:
# library imports
import pandas as pd
import re

## Data Cleaning

In [27]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/giovani-gutierrez/animal_shelter_outcomes/refs/heads/main/data/raw.csv"
)

In [28]:
# set all column names to be in snake case
for col in df.columns:
    new_name = re.sub("([a-z])([A-Z])", r"\1_\2", col)
    new_name = new_name.replace(" ", "_").lower()
    df.rename(columns={col: new_name}, inplace=True)

# preview new column names
df.columns

Index(['facility', 'animal_id', 'animal_type', 'animal_breed', 'impound_no',
       'admission_fy', 'intake_type', 'intake_group', 'outcome_fy',
       'outcome_type', 'outcome_group', 'intake_date', 'outcome_date',
       'object_id'],
      dtype='str')

In [29]:
# drop unnecessary columns
cols_to_drop = [
    "facility",
    "animal_id",
    "impound_no",
    "admission_fy",
    "outcome_fy",
    "outcome_type",
    "outcome_date",
    "object_id",
]

df = df.drop(columns=cols_to_drop)

In [30]:
# keep only those animals that were alive on intake
df = df[df["intake_group"] != "DECEASED"]
df = df.drop(columns=["intake_group"])

In [31]:
# keep only valid outcome groups
valid_outcomes = ["ADOPTION", "OTHER LIVE", "RESCUE", "RTO", "DIED", "EUTHANASIA"]

df = df[df["outcome_group"].isin(valid_outcomes)]

In [32]:
# map observation to target value
def outcome_map(x):
    if isinstance(x, str):
        x = x.upper()
        if x in ["ADOPTION", "OTHER LIVE", "RESCUE", "RTO"]:
            return "LIVE"
        elif x in ["DIED", "EUTHANASIA"]:
            return "NONLIVE"


df["outcome"] = df["outcome_group"].apply(outcome_map)
df = df.drop(columns=["outcome_group"])

# Feature Engineering & Further Data Cleaning

In [33]:
# create intake month and season columns
df["intake_date"] = pd.to_datetime(df["intake_date"])
df["intake_day"] = df["intake_date"].dt.day
df["intake_month"] = df["intake_date"].dt.month
df["intake_hour"] = df["intake_date"].dt.hour

month2season = {
    1: "WINTER",
    2: "WINTER",
    3: "SPRING",
    4: "SPRING",
    5: "SPRING",
    6: "SUMMER",
    7: "SUMMER",
    8: "SUMMER",
    9: "FALL",
    10: "FALL",
    11: "FALL",
    12: "WINTER",
}

df["intake_season"] = df["intake_month"].map(month2season)

In [34]:
df = df.sort_values(by=["intake_date"], ignore_index=True).drop(columns=["intake_date"])

In [35]:
df.head()

,animal_type,animal_breed,intake_type,outcome,intake_day,intake_month,intake_hour,intake_season
0,DOGS,BEAGLE,STRAY,NONLIVE,21,10,7,FALL
1,OTHER,TORTOISE,STRAY,NONLIVE,19,9,7,FALL
2,DOGS,CHIHUAHUA SH,STRAY,LIVE,3,3,8,SPRING
3,OTHER,TURTLE,STRAY,LIVE,3,10,7,FALL
4,CATS,DOMESTIC SH,STRAY,LIVE,6,5,7,SPRING


In [36]:
# save to csv
df.to_csv(
    "C:/Users/giova/Desktop/PersonalProjects/animal_shelter_outcomes/data/clean.csv",
    index=False,
)